In [1]:
import pandas as pd
# import numpy as np
# import streamlit as st
# from backtesting import Backtest
from backtest import run_backtest
from datetime import datetime, timedelta
import json
# import os
import warnings
yf = __import__("yfinance")  # Import yfinance dynamically

warnings.filterwarnings("ignore")

### Utils functions

In [2]:
# Load Binance precisions file
with open("binance_precisions.json", "r") as f:
    binance_precisions = json.load(f)


# Function to get precision for a given Yahoo Finance symbol
def get_binance_precision(yahoo_symbol):
    """
    Maps Yahoo symbol (e.g., 'BTC-USD') to Binance's trading pair format and retrieves precision for 'BTCUSDT'.
    """
    binance_symbol = (
        yahoo_symbol.replace("-", "").upper() + "T"
    )  # Convert BTC-USD -> BTCUSDT
    return binance_precisions.get(binance_symbol, {}).get("amount_precision", 1)


def get_valid_date_range(timeframe):
    days_limit = TIMEFRAME_LIMITS.get(timeframe, 730)
    end_date = datetime.today()
    start_date = end_date - timedelta(days=days_limit - 1)
    return start_date, end_date


# Load Strategies
def load_strategies():
    with open("str_params.json", "r") as f:
        return json.load(f)["strategies"]

strategies = load_strategies()

# Create a mapping {description: strategy_name}
# strategy_mapping = {v["description"]: k for k, v in strategies.items()}
# Create a mapping {description: strategy_name}
strategy_mapping = {k: v["description"] for k, v in strategies.items()}

# Timeframe mapping for Yahoo Finance
TIMEFRAME_MAPPING = {
    "1m": "1m",
    "5m": "5m",
    "15m": "15m",
    "30m": "30m",
    "1h": "1h",
    "4h": "1h",
    "1d": "1d",
    "1w": "1wk",
    "1M": "1mo",
}

TIMEFRAME_LIMITS = {
    "1m": 60,
    "5m": 60,
    "15m": 60,
    "30m": 60,
    "1h": 730,
    "4h": 730,
    "1d": 730,
    "1w": 1460,
    "1M": 3650,
}

In [3]:
strategy_mapping

{'Strategy 1': 'Reversal Strategy using Bollinger Bands & RSI',
 'Strategy 2': 'Overbought Breakout using RSI & Bollinger Bands',
 'Strategy 3': 'Momentum Strategy using MACD & Bollinger Band Width',
 'Strategy 4': 'Trend Following using 200MA & RSI'}

In [4]:
selected_strategy = "Strategy 4"  # change the number to select a different strategy
selected_description = strategy_mapping[selected_strategy]
selected_strategy

'Strategy 4'

### Initialization

In [5]:
initial_cash = 10000
# spread = 0.1 # spread will be covered in the commission
commission = 0.0000 # 0.05%
position_size = 0.99999999 # 99% of the account balance
strategy_config = strategies[selected_strategy]
indicators = strategy_config["indicators"]

data_source = "Yahoo Finance"
symbol = "BTC-USD"
timeframe = "1h"
trade_mode = "long" # default to both, can be changed to long or short

# Convert symbol to Binance format and fetch precision amount
precision_amount = get_binance_precision(symbol)

In [6]:
updated_indicators = {}

for key, value in indicators.items():
    if isinstance(value, dict):
        updated_indicators[key] = value["default"]
    else:
        updated_indicators[key] = value

In [7]:
start_date, end_date = get_valid_date_range(timeframe)
print(f"Fetching data from {start_date} to {end_date}")
print(f"Selected Symbol: {symbol}")

df = None  # Initialize dataframe
if data_source == "Yahoo Finance":
    try:
        df = yf.download(
            symbol,
            start=start_date,
            end=end_date,
            interval=TIMEFRAME_MAPPING[timeframe],
            auto_adjust=True,
        )

        if df.empty:
            print(
                "Failed to retrieve data. Please check the symbol and timeframe."
            )
        else:
            df.reset_index(inplace=True)

            # Flatten MultiIndex if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(
                    0
                )  # Get second level (actual names)
            # Rename column if needed (1D timeframe has "Date", intraday has "Datetime")
            if "Datetime" in df.columns:
                df.rename(columns={"Datetime": "Date"}, inplace=True)
            df.columns = ["Date", "Open", "High", "Low", "Close", "Volume"]
            required_columns = ["Date", "Open", "High", "Low", "Close", "Volume"]
            missing_columns = [
                col for col in required_columns if col not in df.columns
            ]

            if missing_columns:
                print(f"Missing columns in data: {missing_columns}")
            else:
                # Ensure all required columns are present
                df = df[required_columns]

            df.set_index("Date", inplace=True)
            df.index = pd.to_datetime(
                df.index
            )  # Ensure Date column is in datetime format

            print("Data fetched successfully!")
            print(df.tail())

    except Exception as e:
        print(f"Error fetching data: {e}")


Fetching data from 2023-04-01 10:13:56.803587 to 2025-03-30 10:13:56.803587
Selected Symbol: BTC-USD


[*********************100%***********************]  1 of 1 completed

Data fetched successfully!
                                   Open          High           Low  \
Date                                                                  
2025-03-30 01:00:00+00:00  82907.812500  83011.921875  82774.390625   
2025-03-30 02:00:00+00:00  83287.250000  83316.828125  82917.070312   
2025-03-30 03:00:00+00:00  83045.812500  83456.398438  83045.812500   
2025-03-30 04:00:00+00:00  83084.210938  83268.781250  83056.195312   
2025-03-30 05:00:00+00:00  83091.851562  83103.273438  83068.296875   

                                  Close      Volume  
Date                                                 
2025-03-30 01:00:00+00:00  82774.390625   191179776  
2025-03-30 02:00:00+00:00  82917.070312           0  
2025-03-30 03:00:00+00:00  83294.437500  1365654528  
2025-03-30 04:00:00+00:00  83056.195312  1142220800  
2025-03-30 05:00:00+00:00  83102.898438   372587520  


In [8]:
df['2023-04-09 20:00:00':'2023-04-11 07:00:00']

,Open,High,Low,Close,Volume
Date,,,,,
2023-04-09 20:00:00+00:00,28121.792969,28168.503906,28110.191406,28143.519531,22500352
2023-04-09 21:00:00+00:00,28455.757812,28457.791016,28124.107422,28124.107422,641861632
2023-04-09 22:00:00+00:00,28382.580078,28532.830078,28380.978516,28483.744141,499735552
2023-04-09 23:00:00+00:00,28348.171875,28426.322266,28328.011719,28375.900391,201065472
2023-04-10 00:00:00+00:00,28402.212891,28422.853516,28315.357422,28336.027344,90988544
2023-04-10 01:00:00+00:00,28322.990234,28401.011719,28281.267578,28401.011719,159847424
2023-04-10 02:00:00+00:00,28330.236328,28357.644531,28292.349609,28318.179688,0
2023-04-10 03:00:00+00:00,28295.853516,28335.802734,28289.373047,28321.570312,83016704
2023-04-10 04:00:00+00:00,28243.388672,28323.544922,28243.388672,28296.113281,0


## Backtesting the Strategy

In [9]:
if df is not None and not df.empty:
    strategy_config["initial_cash"] = initial_cash
    # strategy_config["initial_cash"] = max(
    #     strategy_config["initial_cash"], df["Close"].max()
    # )
    strategy_config["position_size"] = position_size
    strategy_config["indicators"] = updated_indicators
    # strategy_config["spread"] = spread
    strategy_config["commission"] = commission
    strategy_config["precision_amount"] = precision_amount
    strategy_config["symbol"] = symbol
    strategy_config["timeframe"] = timeframe
    strategy_config["trade_mode"] = trade_mode
    
    max_price = df['Close'].max()

    if initial_cash < max_price: # making sure the we can always buy 1000 units maximum
        position_size_factor = 1000
    elif initial_cash < max_price * 10:
        position_size_factor = 100
    elif initial_cash < max_price * 100:
        position_size_factor = 10
    else:
        position_size_factor = 1  # Full position size


    stats = run_backtest(df / position_size_factor, selected_strategy, strategy_config)
    print("Backtest Results")
    print(stats)

📊 Running backtest with strategy: Strategy 4
📌 Strategy parameters: {'description': 'Trend Following using 200MA & RSI', 'indicators': {'sma_length': 200, 'rsi_length': 14, 'rsi_threshold': 50, 'take_profit_pct': 0.05, 'stop_loss_pct': 0.05}, 'initial_cash': 10000, 'position_size': 0.99999999, 'commission': 0.0, 'precision_amount': 0.0001, 'symbol': 'BTC-USD', 'timeframe': '1h', 'trade_mode': 'long'}
Backtest Results
Start                     2023-04-01 10:00...
End                       2025-03-30 05:00...
Duration                    728 days 19:00:00
Exposure Time [%]                    72.24105
Equity Final [$]                  18690.23746
Equity Peak [$]                   27893.65352
Return [%]                           86.90237
Buy & Hold Return [%]               197.68372
Return (Ann.) [%]                     36.7709
Volatility (Ann.) [%]                57.07995
CAGR [%]                             36.78314
Sharpe Ratio                           0.6442
Sortino Ratio              

### Trades

In [26]:
_trades = stats._trades
trades = _trades.copy() # Create a copy of the trades DataFrame, to ensure base trades DataFrame is not modified
precision = len(str(df['Close'].iloc[-1]).split('.')[1])
trades["EntryPrice"] = (
    trades["EntryPrice"].astype(float).apply(lambda x: format(x * position_size_factor, f".{precision}f"))
)
trades["ExitPrice"] = (
    trades["ExitPrice"].astype(float).apply(lambda x: format(x * position_size_factor, f".{precision}f"))
)


trades['SL'] = trades['SL'] * position_size_factor
trades['TP'] = trades['TP'] * position_size_factor

trades['Size'] = trades['Size'] / position_size_factor

# drop Entry_λ(C) and Exit_λ(C) columns, if exists
for col in ['Entry_λ(C)', 'Exit_λ(C)']:
    if col in trades.columns:
        trades.drop(columns=[col], inplace=True)

trades

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,ReturnPct,EntryTime,ExitTime,Duration,Tag
0,0.347,203,230,28455.7578125,29898.2851562,27032.969922,29878.545703,500.556988,0.050694,2023-04-09 21:00:00+00:00,2023-04-11 00:00:00+00:00,1 days 03:00:00,None
1,0.344,231,445,30207.0898438,28696.7353516,NaN,31717.444336,-519.561945,-0.050000,2023-04-11 01:00:00+00:00,2023-04-19 23:00:00+00:00,8 days 22:00:00,None
2,0.329,548,555,29995.8378906,27957.8261719,NaN,31495.629785,-670.505855,-0.067943,2023-04-26 12:00:00+00:00,2023-04-26 19:00:00+00:00,0 days 07:00:00,None
3,0.325,556,578,28315.9570312,29834.0488281,26900.159180,29731.754883,493.379834,0.053613,2023-04-26 20:00:00+00:00,2023-04-27 18:00:00+00:00,0 days 22:00:00,None
4,0.326,579,671,29685.2304688,28185.2070312,NaN,31169.491992,-489.007641,-0.050531,2023-04-27 19:00:00+00:00,2023-05-01 15:00:00+00:00,3 days 20:00:00,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,0.228,16798,16826,87414.5781250,91785.3070313,83043.849219,91785.307031,996.526191,0.050000,2025-03-04 21:00:00+00:00,2025-03-06 01:00:00+00:00,1 days 04:00:00,None
116,0.228,16827,16849,91830.1250000,85555.5703125,NaN,96421.631250,-1430.598469,-0.068328,2025-03-06 02:00:00+00:00,2025-03-07 00:00:00+00:00,0 days 22:00:00,None
117,0.222,16850,16911,88077.8828125,83594.8750000,NaN,92481.776953,-995.227734,-0.050898,2025-03-07 01:00:00+00:00,2025-03-09 14:00:00+00:00,2 days 13:00:00,None
118,0.220,17034,17271,84346.9921875,88564.3417969,80129.642578,88564.341797,927.816914,0.050000,2025-03-14 17:00:00+00:00,2025-03-24 14:00:00+00:00,9 days 21:00:00,None


In [15]:
df['2023-04-26 19:00:00':'2023-04-27 18:00:00']

,Open,High,Low,Close,Volume
Date,,,,,
2023-04-26 19:00:00+00:00,27957.826172,29732.888672,27902.912109,29732.019531,2563819520
2023-04-26 20:00:00+00:00,28315.957031,28315.957031,27324.548828,27942.675781,2541881344
2023-04-26 21:00:00+00:00,28625.640625,28625.640625,28147.333984,28380.500000,426708992
2023-04-26 22:00:00+00:00,28307.880859,28686.220703,28229.384766,28686.220703,416958464
2023-04-26 23:00:00+00:00,28417.242188,28450.632812,28290.832031,28305.537109,0
2023-04-27 00:00:00+00:00,29294.464844,29452.402344,28402.886719,28428.464844,1673949184
2023-04-27 01:00:00+00:00,28735.347656,29326.994141,28584.007812,29268.369141,764401664
2023-04-27 02:00:00+00:00,29054.978516,29069.574219,28744.066406,28744.066406,590315520
2023-04-27 03:00:00+00:00,28879.662109,29047.890625,28861.431641,29047.890625,74919936


In [14]:
28315.9570312 * 1.05

29731.75488276